In [1]:
! pip install datasets

In [3]:
from datasets import load_dataset
dataset =  load_dataset("bentrevett/multi30k")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.jsonl: 0.00B [00:00, ?B/s]

val.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [4]:
train,val,test = dataset["train"],dataset["validation"],dataset["test"]

## Building the tokenization

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
import spacy
import datasets
import tqdm

In [7]:
seed = 1234

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

In [8]:
train[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

In [11]:
!python -m spacy download en_core_web_sm
!python -m spacy download de_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 112.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 71.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [12]:
## loading the tokenizer
en_nlp = spacy.load("en_core_web_sm")
de_nlp = spacy.load("de_core_news_sm")

In [14]:
sentence = "hi how are you"
[text.text for text in en_nlp.tokenizer(sentence)]

['hi', 'how', 'are', 'you']

In [27]:
def tokenize_examples(example,en_nlp,de_nlp,lower,sos_token,eos_token,max_length):
  english = [token.text for token in en_nlp.tokenizer(example["en"])][:max_length]
  german = [token.text  for token in de_nlp.tokenizer(example["de"])][:max_length]
  if lower:
    english = [text.lower() for text in english]
    german = [text.lower() for text in german]
  english = [sos_token] + english + [eos_token]
  german = [sos_token] + german + [eos_token]
  return {"en_tokens":english,"de_tokens":german}

In [28]:
tokenize_examples(train[0],en_nlp,de_nlp,1,"<sos>","<eos>",100)

{'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

In [29]:
max_length = 1_000
lower = True
sos_token = "<sos>"
eos_token = "<eos>"

fn_kwargs = {
    "en_nlp": en_nlp,
    "de_nlp": de_nlp,
    "lower": lower,
    "sos_token": sos_token,
    "eos_token": eos_token,
    "max_length": max_length
}

In [30]:
train_data = train.map(tokenize_examples, fn_kwargs=fn_kwargs)
valid_data = val.map(tokenize_examples, fn_kwargs=fn_kwargs)
test_data = test.map(tokenize_examples, fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Vocabulary

In [31]:
## so lets build the vocabulary
unk_token = "<unk>" # to capture the unknown token while inference
pad_token = "<pad>" # to make the len same
special_tokens = [
    unk_token,
    pad_token,
    sos_token,
    eos_token,
]

In [35]:
train_data["en_tokens"][0]

['<sos>',
 'two',
 'young',
 ',',
 'white',
 'males',
 'are',
 'outside',
 'near',
 'many',
 'bushes',
 '.',
 '<eos>']

In [42]:
from collections import Counter
vocab = {
    "<unk>":0,
    "<pad>":1,
    "<sos>":2,
    "<eos>":3
}

In [37]:
Counter

collections.Counter

In [43]:
for sentence in train_data["en_tokens"]:
  for token in sentence:
    if token not in vocab:
      vocab[token] = len(vocab) + 1
  break

In [39]:
sentence

['<sos>',
 'two',
 'young',
 ',',
 'white',
 'males',
 'are',
 'outside',
 'near',
 'many',
 'bushes',
 '.',
 '<eos>']

In [44]:
vocab

{'<unk>': 0,
 '<pad>': 1,
 '<sos>': 2,
 '<eos>': 3,
 'two': 5,
 'young': 6,
 ',': 7,
 'white': 8,
 'males': 9,
 'are': 10,
 'outside': 11,
 'near': 12,
 'many': 13,
 'bushes': 14,
 '.': 15}

In [46]:
en_vocab =  {
    "<unk>":0,
    "<pad>":1,
    "<sos>":2,
    "<eos>":3
}
# english
en_counter = Counter()
for sentence in train_data["en_tokens"]:
    en_counter.update(sentence)
min_freq = 2
for token, count in en_counter.items():
    if count >= min_freq and token not in en_vocab:
        en_vocab[token] = len(en_vocab)

In [47]:
en_vocab

{'<unk>': 0,
 '<pad>': 1,
 '<sos>': 2,
 '<eos>': 3,
 'two': 4,
 'young': 5,
 ',': 6,
 'white': 7,
 'males': 8,
 'are': 9,
 'outside': 10,
 'near': 11,
 'many': 12,
 'bushes': 13,
 '.': 14,
 'several': 15,
 'men': 16,
 'in': 17,
 'hard': 18,
 'hats': 19,
 'operating': 20,
 'a': 21,
 'giant': 22,
 'pulley': 23,
 'system': 24,
 'little': 25,
 'girl': 26,
 'climbing': 27,
 'into': 28,
 'wooden': 29,
 'playhouse': 30,
 'man': 31,
 'blue': 32,
 'shirt': 33,
 'is': 34,
 'standing': 35,
 'on': 36,
 'ladder': 37,
 'cleaning': 38,
 'window': 39,
 'at': 40,
 'the': 41,
 'stove': 42,
 'preparing': 43,
 'food': 44,
 'green': 45,
 'holds': 46,
 'guitar': 47,
 'while': 48,
 'other': 49,
 'observes': 50,
 'his': 51,
 'smiling': 52,
 'stuffed': 53,
 'lion': 54,
 'trendy': 55,
 'talking': 56,
 'her': 57,
 'cellphone': 58,
 'gliding': 59,
 'slowly': 60,
 'down': 61,
 'street': 62,
 'woman': 63,
 'with': 64,
 'large': 65,
 'purse': 66,
 'walking': 67,
 'by': 68,
 'gate': 69,
 'boys': 70,
 'dancing': 71,
 

In [48]:
de_vocab =  {
    "<unk>":0,
    "<pad>":1,
    "<sos>":2,
    "<eos>":3
}
# german
de_counter = Counter()
for sentence in train_data["de_tokens"]:
    de_counter.update(sentence)
min_freq = 2
for token, count in de_counter.items():
    if count >= min_freq and token not in de_vocab:
        de_vocab[token] = len(de_vocab)

In [51]:
en_vocab["the"]

41

In [52]:
len(en_vocab),len(de_vocab)

(5893, 7853)

In [53]:
# just and test
"The" in en_vocab

False

In [54]:
assert en_vocab[unk_token] == de_vocab[unk_token]
assert en_vocab[pad_token] == de_vocab[pad_token]

unk_index = en_vocab[unk_token]
pad_index = en_vocab[pad_token]
# to check that they got the came index

In [55]:
unk_index , pad_index

(0, 1)

In [57]:
en_vocab["The"] # as this the manual vocab building so some methods to use

KeyError: 'The'

In [92]:
def lookup_indices(example,en_vocab,de_vocab):
  en_indices,de_indices = [],[]
  #english
  for token in example["en_tokens"]:
    if token not in en_vocab:
      en_indices.append(en_vocab["<unk>"])
    else:  # Add this else!
        en_indices.append(en_vocab[token])
  #german
  for token in example["de_tokens"]:
    if token not in de_vocab:
      de_indices.append(de_vocab["<unk>"])
    else:  # Add this else!
        de_indices.append(de_vocab[token])
  return {"en_ids":torch.tensor(en_indices),"de_ids":torch.tensor(de_indices)}

In [93]:
lookup_indices(train_data[0],en_vocab,de_vocab)

{'en_ids': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14,  3]),
 'de_ids': tensor([ 2,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,  3])}

In [90]:
train[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}

In [78]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

In [94]:
fn_kwargs = {
    "en_vocab":en_vocab,
    "de_vocab":de_vocab
}
train_data = train_data.map(lookup_indices,fn_kwargs=fn_kwargs)

Map:   0%|          | 0/29000 [00:00<?, ? examples/s]

In [104]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>'],
 'en_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3],
 'de_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 3]}

In [95]:
valid_data = valid_data.map(lookup_indices,fn_kwargs=fn_kwargs)
test_data = test_data.map(lookup_indices,fn_kwargs=fn_kwargs)

Map:   0%|          | 0/1014 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [69]:
# reverse vocab
en_itos = {i: s for s, i in en_vocab.items()}
de_itos = {i: s for s, i in de_vocab.items()}

def lookup_vocab(example,en_itos,de_itos):
  en_indices,de_indices = [],[]
  #english
  for ids in example["en_ids"]:
    en_indices.append(en_itos[ids])
  #german
  for ids in example["de_ids"]:
    de_indices.append(de_itos[ids])
  return {"en_tokens_rev":(en_indices),"de_tokens_rev":(de_indices)}

In [70]:
train_data

Dataset({
    features: ['en', 'de', 'en_tokens', 'de_tokens'],
    num_rows: 29000
})

In [74]:
lookup_indices(train_data[0],en_vocab,de_vocab)

{'en_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 3],
 'de_ids': [2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 3]}

In [71]:
lookup_vocab(lookup_indices(train_data[0],en_vocab,de_vocab),en_itos,de_itos)

{'en_tokens_rev': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens_rev': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

In [73]:
train_data[0]

{'en': 'Two young, White males are outside near many bushes.',
 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.',
 'en_tokens': ['<sos>',
  'two',
  'young',
  ',',
  'white',
  'males',
  'are',
  'outside',
  'near',
  'many',
  'bushes',
  '.',
  '<eos>'],
 'de_tokens': ['<sos>',
  'zwei',
  'junge',
  'weiße',
  'männer',
  'sind',
  'im',
  'freien',
  'in',
  'der',
  'nähe',
  'vieler',
  'büsche',
  '.',
  '<eos>']}

## Data Loader

In [105]:
def get_collate_fn(pad_index):
    def collate_fn(batch):
        batch_en_ids = [torch.tensor(example["en_ids"]) for example in batch]
        batch_de_ids = [torch.tensor(example["de_ids"]) for example in batch]
        batch_en_ids = nn.utils.rnn.pad_sequence(batch_en_ids, padding_value=pad_index) # padding as the same length for one batch
        batch_de_ids = nn.utils.rnn.pad_sequence(batch_de_ids, padding_value=pad_index)
        batch = {
            "en_ids": batch_en_ids,
            "de_ids": batch_de_ids,
        }
        return batch

    return collate_fn

In [106]:
def get_data_loader(dataset, batch_size, pad_index, shuffle=False):
    collate_fn = get_collate_fn(pad_index)
    data_loader = torch.utils.data.DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=shuffle,
    )
    return data_loader

In [107]:
batch_size = 32

train_data_loader = get_data_loader(train_data, batch_size, pad_index, shuffle=True)
valid_data_loader = get_data_loader(valid_data, batch_size, pad_index)
test_data_loader = get_data_loader(test_data, batch_size, pad_index)

In [108]:
for x in train_data_loader:
  break

In [112]:
len(x["en_ids"]) , len(x["de_ids"])

(22, 22)

## Next model training